In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver, gold
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json


In [2]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [6]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
   json_request = requests.get(url).json()
   return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))
    
bronze_instance = StarWarsBronze(spark, **options)


In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

In [10]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-03 04:07:...|      Luke Skywalker|  1|https://www.swapi...|{"height": "172",...|
|2025-01-03 04:07:...|               C-3PO|  2|https://www.swapi...|{"height": "167",...|
|2025-01-03 04:07:...|               R2-D2|  3|https://www.swapi...|{"height": "96", ...|
|2025-01-03 04:07:...|         Darth Vader|  4|https://www.swapi...|{"height": "202",...|
|2025-01-03 04:07:...|         Leia Organa|  5|https://www.swapi...|{"height": "150",...|
|2025-01-03 04:07:...|           Owen Lars|  6|https://www.swapi...|{"height": "178",...|
|2025-01-03 04:07:...|  Beru Whitesun lars|  7|https://www.swapi...|{"height": "165",...|
|2025-01-03 04:07:...|               R5-D4|  8|https://www.swapi...|{"height": "97", ..

# 2 Silver

In [11]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [13]:
class StarWarsSilver(silver.Silver):    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            sdf = self.transf_people(sdf)
        return sdf

    def transf_people(self, sdf: DataFrame) -> DataFrame:
        sdf = (
            sdf.withColumn("height", sdf.properties.height)
            .withColumn("mass", sdf.properties.mass)
            .withColumn("gender", sdf.properties.gender)
            .drop("url","properties")
        )
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)

In [14]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute("people")

In [15]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 82
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+
|LH_SilverTS               |LH_BronzeTS              |name                 |uid|height |mass   |gender       |
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Cliegg Lars          |62 |183    |unknown|male         |
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Poggle the Lesser    |63 |183    |80     |male         |
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Luminara Unduli      |64 |170    |56.2   |female       |
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Barriss Offee        |65 |166    |50     |female       |
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Dormé                |66 |165    |unknown|female       |
|2025-01-03 04:08:19.218887|2025-01-03 04:07:35.24715|Dooku                |67 |193    |80     |mal

# 3 Gold

In [16]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [17]:
class StarWarsGold(gold.Gold): 
    def people_per_gender(self, sdf: DataFrame) -> DataFrame:
        sdf = sdf.where("gender <> 'n/a'")
        sdf = sdf.where("gender <> 'none'")
        sdf = sdf.groupBy("gender").count()
        return sdf   
    
    def all_females(self, sdf: DataFrame) -> DataFrame:
        return sdf.where("gender = 'female'").drop("LH_SilverTS", "LH_BronzeTS")
    
gold_instance = StarWarsGold(spark, **options)

In [18]:
gold_instance.load(source_tbl="people").transform(tbl_transformations={"peoplegender": "people_per_gender", "peoplefemale": "all_females"}).write(mode="overwrite", merge_schema=True).execute("peoplegender", "peoplefemale")

In [19]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplegender")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 3
+--------------------------+-------------+-----+
|LH_GoldTS                 |gender       |count|
+--------------------------+-------------+-----+
|2025-01-03 04:08:21.366653|female       |17   |
|2025-01-03 04:08:21.366653|male         |60   |
|2025-01-03 04:08:21.366653|hermaphrodite|1    |
+--------------------------+-------------+-----+



In [20]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplefemale")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 17
+--------------------------+------------------+---+------+-------+------+
|LH_GoldTS                 |name              |uid|height|mass   |gender|
+--------------------------+------------------+---+------+-------+------+
|2025-01-03 04:08:22.549695|Luminara Unduli   |64 |170   |56.2   |female|
|2025-01-03 04:08:22.549695|Barriss Offee     |65 |166   |50     |female|
|2025-01-03 04:08:22.549695|Dormé             |66 |165   |unknown|female|
|2025-01-03 04:08:22.549695|Zam Wesell        |70 |168   |55     |female|
|2025-01-03 04:08:22.549695|Taun We           |73 |213   |unknown|female|
|2025-01-03 04:08:22.549695|Jocasta Nu        |74 |167   |unknown|female|
|2025-01-03 04:08:22.549695|R4-P17            |75 |96    |unknown|female|
|2025-01-03 04:08:22.549695|Shaak Ti          |78 |178   |57     |female|
|2025-01-03 04:08:22.549695|Sly Moore         |82 |178   |48     |female|
|2025-01-03 04:08:22.549695|Shmi Skywalker    |43 |163   |unknown|female|
|2025-01-03 04:08:22.5496

# 4 Clean Up

In [21]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")

DataFrame[]